<a href="https://colab.research.google.com/github/genarioazevedoufape/atividades_pet_data_science/blob/main/SAUTER_FORECAST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import datetime as dt

from statsmodels.graphics.tsaplots import plot_acf

import statsmodels.api as sm
import statsmodels.tsa.api as smt
from statsmodels.tsa.seasonal import seasonal_decompose

import warnings
warnings.filterwarnings('ignore')
!pip install mlforecast
!pip install statsforecast

### Importação de dados

In [ ]:
df = pd.read_parquet('/content/drive/MyDrive/Datasets/M5_full.parquet')

In [ ]:
df

,id,item_id,dept_id,cat_id,store_id,state_id,value,date
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,2011-01-29
...,...,...,...,...,...,...,...,...
59181085,FOODS_3_823_WI_3_evaluation,FOODS_3_823,FOODS_3,FOODS,WI_3,WI,1,2016-05-22
59181086,FOODS_3_824_WI_3_evaluation,FOODS_3_824,FOODS_3,FOODS,WI_3,WI,0,2016-05-22
59181087,FOODS_3_825_WI_3_evaluation,FOODS_3_825,FOODS_3,FOODS,WI_3,WI,2,2016-05-22
59181088,FOODS_3_826_WI_3_evaluation,FOODS_3_826,FOODS_3,FOODS,WI_3,WI,0,2016-05-22


In [ ]:
df.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'value',
       'date'],
      dtype='object')

In [ ]:
df.id.describe()

count                        59181090
unique                          30490
top       FOODS_1_001_CA_1_evaluation
freq                             1941
Name: id, dtype: object

In [ ]:
df.date = pd.to_datetime(df.date)
# df.set_index('date', inplace=True)
# df.reset_index(inplace=True)

### Trabalhando com a coluna id

In [ ]:
df_id_mensal = df.groupby(['id']).resample('M', on='date')['value'].sum().reset_index()

In [ ]:
df_id_mensal

,id,date,value
0,FOODS_1_001_CA_1_evaluation,2011-01-31,3
1,FOODS_1_001_CA_1_evaluation,2011-02-28,40
2,FOODS_1_001_CA_1_evaluation,2011-03-31,40
3,FOODS_1_001_CA_1_evaluation,2011-04-30,23
4,FOODS_1_001_CA_1_evaluation,2011-05-31,50
...,...,...,...
1981845,HOUSEHOLD_2_516_WI_3_evaluation,2016-01-31,3
1981846,HOUSEHOLD_2_516_WI_3_evaluation,2016-02-29,2
1981847,HOUSEHOLD_2_516_WI_3_evaluation,2016-03-31,2
1981848,HOUSEHOLD_2_516_WI_3_evaluation,2016-04-30,1


### Criação de novas series

In [ ]:
def buscar_por_categoria(df, departamento):
    return df[df['id'].str.contains(departamento, regex=True)]

# Exemplos de uso:
df_foods = buscar_por_categoria(df_id_mensal, 'FOODS')
df_household = buscar_por_categoria(df_id_mensal, 'HOUSEHOLD')
df_hobbies = buscar_por_categoria(df_id_mensal, 'HOBBIES')

In [ ]:
def buscar(df, cats): # por estado
    cat_regex = '|'.join(cats)
    return df[df['id'].str.contains(cat_regex, regex=True)]

cats = ['CA', 'TX', 'WI']
df_cats = buscar(df, cats)

In [ ]:
df_ca = df_cats[df_cats['id'].str.contains('CA', regex=False)]
df_tx = df_cats[df_cats['id'].str.contains('TX', regex=False)]
df_wi = df_cats[df_cats['id'].str.contains('WI', regex=False)]

In [ ]:
def buscar(df, cats): #por categoria
    cat_regex = '|'.join(cats)
    return df[df['id'].str.contains(cat_regex, regex=True)]

cats = ['HOUSEHOLD', 'FOODS', 'HOBBIES']
df_cats = buscar(df, cats)

In [ ]:
df_household = df_cats[df_cats['id'].str.contains('HOUSEHOLD', regex=False)]
df_foods = df_cats[df_cats['id'].str.contains('FOODS', regex=False)]
df_hobbies = df_cats[df_cats['id'].str.contains('HOBBIES', regex=False)]

In [ ]:
def buscar(df, cats): # por loja
    cat_regex = '|'.join(cats)
    return df[df['id'].str.contains(cat_regex, regex=True)]

cats = ['CA_1','CA_2', 'CA_3', 'CA_4','TX_1', 'TX_2', 'TX_3','WI_1', 'WI_2', 'WI_3']
df_cats = buscar(df, cats)

In [ ]:
df_ca_1 = df_cats[df_cats['id'].str.contains('CA_1', regex=False)]
df_ca_2 = df_cats[df_cats['id'].str.contains('CA_2', regex=False)]
df_ca_3 = df_cats[df_cats['id'].str.contains('CA_3', regex=False)]
df_ca_4 = df_cats[df_cats['id'].str.contains('CA_4', regex=False)]
df_tx_1 = df_cats[df_cats['id'].str.contains('TX_1', regex=False)]
df_tx_2 = df_cats[df_cats['id'].str.contains('TX_2', regex=False)]
df_tx_3 = df_cats[df_cats['id'].str.contains('TX_3', regex=False)]
df_wi_1 = df_cats[df_cats['id'].str.contains('WI_1', regex=False)]
df_wi_2 = df_cats[df_cats['id'].str.contains('WI_2', regex=False)]
df_wi_3 = df_cats[df_cats['id'].str.contains('WI_3', regex=False)]

In [ ]:
df_ca_3.id.nunique()

3049

In [ ]:
df_ca_1.id.nunique()

3049

In [ ]:
df_ca_2.id.nunique()

3049

In [ ]:
df_ca_4.id.nunique()

3049

### FORECAST

#### Previsão hobbies mensal

In [ ]:
df_hobbies = pd.DataFrame(
    {
        'ds' : df_hobbies.date,
        'y' : df_hobbies.value,
        'unique_id' : df_hobbies.id
    }
)

In [ ]:
df_hobbies

,ds,y,unique_id
934050,2011-01-31,0,HOBBIES_1_001_CA_1_evaluation
934051,2011-02-28,0,HOBBIES_1_001_CA_1_evaluation
934052,2011-03-31,0,HOBBIES_1_001_CA_1_evaluation
934053,2011-04-30,0,HOBBIES_1_001_CA_1_evaluation
934054,2011-05-31,0,HOBBIES_1_001_CA_1_evaluation
...,...,...,...
1301295,2016-01-31,16,HOBBIES_2_149_WI_3_evaluation
1301296,2016-02-29,15,HOBBIES_2_149_WI_3_evaluation
1301297,2016-03-31,11,HOBBIES_2_149_WI_3_evaluation
1301298,2016-04-30,5,HOBBIES_2_149_WI_3_evaluation


In [ ]:
test_size = 0.2

def split_train_test(group):
    split_index = int(len(group) * (1 - test_size))
    return group.iloc[:split_index], group.iloc[split_index:]

train_data, test_data = [], []
for _, group in df_hobbies.groupby('unique_id'):
    train, test = split_train_test(group)
    train_data.append(train)
    test_data.append(test)

train = pd.concat(train_data)
test = pd.concat(test_data)
h = test['ds'].nunique()

##### MLforecast

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from xgboost import XGBRegressor
from sklearn.svm import SVR
from mlforecast import MLForecast
from sklearn.linear_model import LinearRegression
from window_ops.rolling import rolling_mean, rolling_max, rolling_min

models = [XGBRegressor(random_state=0, n_estimators=50), SVR(C = 1, epsilon = 0.1, kernel = 'linear'), LinearRegression(fit_intercept=False)]

model = MLForecast(models=models,
                   freq='M',
                   lags=[12, 24, 36],
                   lag_transforms={
                       1: [(rolling_mean, 12), (rolling_max, 12), (rolling_min, 12)],
                   },
                   date_features=['month'],
                   num_threads=6)


model.fit(train, id_col='unique_id', time_col='ds', target_col='y', static_features=[])

In [ ]:
forecast = model.predict(h)
forecast = forecast.merge(test[['ds', 'y', 'unique_id']], on=['unique_id', 'ds'], how='left')

In [ ]:
forecast.head()

In [ ]:
metrics = {}
model_names = ['XGBRegressor', 'SVR',	'LinearRegression']

for model_name in model_names:
    y =forecast['y']
    y_pred = forecast[model_name]

    mae = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mape = mean_absolute_percentage_error(y, y_pred)
    r2 = r2_score(y, y_pred)

    metrics[model_name] = {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2
    }

# Imprimir métricas
for model_name, metric in metrics.items():
    print(f"--------------- Métricas para {model_name} ---------------")
    print(f"MAE: {metric['MAE']}")
    print(f"RMSE: {metric['RMSE']}")
    print(f"MAPE: {metric['MAPE']}")
    print(f"R2: {metric['R2']}\n")

In [ ]:
# Assuming 'p' is your DataFrame containing data for multiple unique_ids

for device in forecast['unique_id'].unique():
    p_device = forecast.loc[forecast['unique_id'] == device]

    fig, ax = plt.subplots(3, 1, figsize=(16, 10))
    models = [('XGBRegressor', 'XGBRegressor Predicted'), ('SVR', 'SVR Predicted'), ('LinearRegression', 'LinearRegression Predicted')]
    for i, (model_col, label) in enumerate(models):
        ax[i].plot(p_device['ds'], p_device[model_col], label=label)
        ax[i].plot(p_device['ds'], p_device['y'], label='Actual', linestyle='--')
        ax[i].set_title(f'{label} - {device}')  # Include unique_id in the title
        ax[i].set_xlabel('Date')
        ax[i].set_ylabel('Sessions')
        ax[i].legend()

    plt.tight_layout()
    plt.show()

##### Statsforecast